<a href="https://colab.research.google.com/github/RatchanonPa/Data-Warehouse-and-Big-Data-Analytics/blob/main/M2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 91.8 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

## Mask Language Model

In [4]:
from transformers import AutoModelForMaskedLM, AutoTokenizer
import torch

In [5]:
model_name = 'google-t5/t5-small'

In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name, device_map="auto")

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

ValueError: Unrecognized configuration class <class 'transformers.models.t5.configuration_t5.T5Config'> for this kind of AutoModel: AutoModelForMaskedLM.
Model type should be one of AlbertConfig, BartConfig, BertConfig, BigBirdConfig, CamembertConfig, ConvBertConfig, Data2VecTextConfig, DebertaConfig, DebertaV2Config, DistilBertConfig, ElectraConfig, ErnieConfig, EsmConfig, FlaubertConfig, FNetConfig, FunnelConfig, IBertConfig, LayoutLMConfig, LongformerConfig, LukeConfig, MBartConfig, MegaConfig, MegatronBertConfig, MobileBertConfig, ModernBertConfig, MPNetConfig, MraConfig, MvpConfig, NezhaConfig, NystromformerConfig, PerceiverConfig, QDQBertConfig, ReformerConfig, RemBertConfig, RobertaConfig, RobertaPreLayerNormConfig, RoCBertConfig, RoFormerConfig, SqueezeBertConfig, TapasConfig, Wav2Vec2Config, XLMConfig, XLMRobertaConfig, XLMRobertaXLConfig, XmodConfig, YosoConfig.

In [ ]:
mask = tokenizer.mask_token
mask

'<mask>'

In [ ]:
text = f'สวัสดี{mask}'
text

'สวัสดี<mask>'

In [ ]:
model_inputs = tokenizer(text, return_tensors="pt").to(model.device)
model_inputs.keys()

dict_keys(['input_ids', 'attention_mask'])

In [ ]:
with torch.no_grad():
    outputs = model(**model_inputs)

In [ ]:
mask_token_index = (model_inputs.input_ids == tokenizer.mask_token_id).nonzero(as_tuple=True)[1]
logits = outputs.logits
masked_token_logits = logits[0, mask_token_index, :]

In [ ]:
top_k = 5
top_logits, top_tokens = torch.topk(masked_token_logits, top_k, dim=1)
predicted_words = tokenizer.convert_ids_to_tokens(top_tokens[0])
probs = torch.softmax(top_logits, dim=-1)
for i, (word, prob) in enumerate(zip(predicted_words, probs[0])):
    print(f"Prediction {i+1} ({prob:.2f}): {text.replace(tokenizer.mask_token, word)}")

Prediction 1 (0.54): สวัสดีครับ
Prediction 2 (0.24): สวัสดีค่ะ
Prediction 3 (0.18): สวัสดีปีใหม่
Prediction 4 (0.02): สวัสดีวันอังคาร
Prediction 5 (0.02): สวัสดีวันจันทร์


In [13]:
!unzip /content/superai5-esan-to-thai-machine-translation.zip

Archive:  /content/superai5-esan-to-thai-machine-translation.zip
  inflating: sample_submission.csv   
  inflating: submission.csv          
  inflating: test.csv                
  inflating: train.csv               
  inflating: val.csv                 


In [22]:
pip install pandas transformers[sentencepiece] datasets pyThaiNLP

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.3/19.3 MB 96.5 MB/s eta 0:00:00


In [34]:
import pandas as pd
from datasets import Dataset

# Load datasets
train_df = pd.read_csv("/content/train.csv")
test_df = pd.read_csv("/content/test.csv")
val_df = pd.read_csv("/content/val.csv")

# Convert to Hugging Face Dataset format
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

In [25]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/mt5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

/usr/local/lib/python3.11/dist-packages/transformers/convert_slow_tokenizer.py:561: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [35]:
max_input_length = 128
max_target_length = 128

def preprocess_function(examples):
    inputs = [ex for ex in examples["input"]]
    targets = [ex for ex in examples["output"]]

    # Tokenize inputs
    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
        padding="max_length"
    )

    # Tokenize targets
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            targets,
            max_length=max_target_length,
            truncation=True,
            padding="max_length"
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_val = val_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/1943 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:3961: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/112 [00:00<?, ? examples/s]

In [40]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="esan-thai-translator",
    evaluation_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    save_total_limit=2,
    predict_with_generate=True,
    fp16=True,
    report_to="none",  # เพิ่มบรรทัดนี้เพื่อปิด WandB
    run_name="translation-run-01",  # ตั้งชื่อ run ให้แตกต่างจาก output_dir
)

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [41]:

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

trainer.train()

<ipython-input-41-8bfb33112cb0>:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss
1,No log,nan
2,No log,nan
3,0.000000,nan
4,0.000000,nan
5,0.000000,nan


TrainOutput(global_step=1215, training_loss=0.0, metrics={'train_runtime': 354.2917, 'train_samples_per_second': 27.421, 'train_steps_per_second': 3.429, 'total_flos': 1284200811724800.0, 'train_loss': 0.0, 'epoch': 5.0})

In [42]:
def generate_translation(batch):
    inputs = tokenizer(
        batch["input"],
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(**inputs, max_length=128)
    return {"predicted": tokenizer.batch_decode(outputs, skip_special_tokens=True)}

test_results = test_dataset.map(generate_translation, batched=True, batch_size=8)

Parameter 'function'=<function generate_translation at 0x79ef6c63c7c0> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Map:   0%|          | 0/374 [00:00<?, ? examples/s]

In [43]:
submission_df = pd.DataFrame({
    "id": test_df["id"],
    "output": test_results["predicted"]
})

# Ensure proper tokenization with PyThaiNLP
from pythainlp.tokenize import word_tokenize
submission_df["output"] = submission_df["output"].apply(
    lambda x: " ".join(word_tokenize(x, engine="newmm"))
)

submission_df.to_csv("nlp_submission.csv", index=False)